# Phát hiện đánh nhau: huấn luyện trên Kaggle

Trước khi chạy, trong phần **Settings** ở cột phải của notebook:

1. **Input → Add Input:** thêm bộ dữ liệu `vulamnguyen/rwf2000`.
2. **Accelerator:** chọn GPU.
3. **Internet:** bật On (cần xác minh số điện thoại Kaggle), để cài Ultralytics và tải trọng số.

Sau đó chọn **Save Version → Save & Run All** để notebook chạy nền, không cần giữ trình duyệt mở. Kết quả nằm ở tab **Output**, file `results.zip`.

File này được sinh bởi `tools/build_kaggle_notebook.py`, nên sửa code trong `src/` rồi sinh lại, không sửa trực tiếp ở đây.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

try:
    import ultralytics
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)

def find_dataset():
    if os.environ.get('FIGHT_ROOT'):
        return Path(os.environ['FIGHT_ROOT'])
    for train in Path('/kaggle/input').rglob('train'):
        if (train / 'Fight').is_dir() and (train.parent / 'val').is_dir():
            return train.parent
    raise FileNotFoundError('Khong thay RWF-2000. Hay them bo vulamnguyen/rwf2000 vao Input.')

ROOT = find_dataset()
WORK = Path(os.environ.get('FIGHT_WORK', '/kaggle/working'))
# Frame chi la du lieu trung gian nen de ngoai thu muc output cua Kaggle.
DATA = Path(os.environ.get('FIGHT_DATA', '/tmp/fight'))
EPOCHS = int(os.environ.get('FIGHT_EPOCHS', 30))
IMAGE_SIZE = int(os.environ.get('FIGHT_IMAGE_SIZE', 224))
EXTRA = ['--no-pretrained'] if os.environ.get('FIGHT_NO_PRETRAINED') else []
SRC = WORK / 'src'
SRC.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
print('Du lieu:', ROOT)

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG CO, hay bat GPU trong Settings')

def run(script, *args):
    subprocess.run([sys.executable, str(SRC / script), *map(str, args)], check=True)


In [ ]:
(SRC / 'crop_persons.py').write_text('"""Dung YOLO11 phat hien nguoi, cat vung bao tat ca moi nguoi trong frame (dau vao cua M3).\n\n    python src/crop_persons.py --frames data/frames --out data/crops\n"""\nimport argparse\nfrom pathlib import Path\n\nimport cv2\n\nfrom prepare_data import read_image, write_image\n\n\ndef union_box(boxes, width, height, margin=0.1):\n    """Hop cac box (x1, y1, x2, y2), noi rong theo margin va gioi han trong anh."""\n    if len(boxes) == 0:\n        return None\n    x1 = min(b[0] for b in boxes)\n    y1 = min(b[1] for b in boxes)\n    x2 = max(b[2] for b in boxes)\n    y2 = max(b[3] for b in boxes)\n    pad_x, pad_y = (x2 - x1) * margin, (y2 - y1) * margin\n    return (\n        max(0, round(x1 - pad_x)),\n        max(0, round(y1 - pad_y)),\n        min(width, round(x2 + pad_x)),\n        min(height, round(y2 + pad_y)),\n    )\n\n\ndef crop_frames_dir(in_dir, out_dir, detector, size=224, margin=0.1):\n    """Cat moi frame trong mot clip; khong co nguoi thi giu ca frame. Bo qua frame da cat truoc do."""\n    stats = {"frames": 0, "no_person": 0}\n    for frame_path in sorted(Path(in_dir).glob("*.jpg")):\n        target = Path(out_dir) / frame_path.name\n        if target.exists():\n            continue\n        image = read_image(frame_path)\n        height, width = image.shape[:2]\n        box = union_box(detector(image), width, height, margin)\n        stats["frames"] += 1\n        if box is None or box[2] <= box[0] or box[3] <= box[1]:\n            stats["no_person"] += 1\n        else:\n            image = image[box[1] : box[3], box[0] : box[2]]\n        write_image(target, cv2.resize(image, (size, size)))\n    return stats\n\n\ndef make_yolo_detector(weights="yolo11n.pt", conf=0.25, imgsz=640):\n    from ultralytics import YOLO\n\n    model = YOLO(weights)\n\n    def detect(image):\n        result = model.predict(image, classes=[0], conf=conf, imgsz=imgsz, verbose=False)[0]\n        return result.boxes.xyxy.cpu().tolist()\n\n    return detect\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--frames", default="data/frames")\n    parser.add_argument("--out", default="data/crops")\n    parser.add_argument("--weights", default="yolo11n.pt")\n    parser.add_argument("--size", type=int, default=224)\n    parser.add_argument("--conf", type=float, default=0.25)\n    args = parser.parse_args()\n\n    detector = make_yolo_detector(args.weights, args.conf)\n    frames_root = Path(args.frames)\n    total = {"frames": 0, "no_person": 0}\n    for clip_dir in sorted(p for p in frames_root.glob("*/*/*") if p.is_dir()):\n        stats = crop_frames_dir(clip_dir, Path(args.out) / clip_dir.relative_to(frames_root), detector, args.size)\n        total = {key: total[key] + stats[key] for key in total}\n    print(f"Da cat {total[\'frames\']} frame, {total[\'no_person\']} frame khong co nguoi (giu nguyen ca frame)")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8');

In [ ]:
(SRC / 'eda.py').write_text('"""EDA cho bo du lieu video. Doc splits.csv tu prepare_data.py, ghi bang va bieu do vao --out.\n\n    python src/eda.py --root D:/RWF-2000 --splits splits/splits.csv --out results/eda\n"""\nimport argparse\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport cv2\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\n\nfrom prepare_data import read_frames, sample_indices\n\nLABEL_COLORS = {"Fight": "#c8553d", "NonFight": "#3d6fc8"}\n\n\ndef frame_quality(frame):\n    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)\n    return {\n        "brightness": float(gray.mean()),\n        "contrast": float(gray.std()),\n        "sharpness": float(cv2.Laplacian(gray, cv2.CV_64F).var()),\n    }\n\n\ndef _motion(frames):\n    grays = [cv2.cvtColor(f, cv2.COLOR_BGR2GRAY).astype(np.float32) for f in frames]\n    return float(np.mean([np.abs(a - b).mean() for a, b in zip(grays, grays[1:])])) if len(grays) > 1 else 0.0\n\n\ndef motion_score(path):\n    """Chenh lech trung binh giua hai frame lien tiep, xap xi do lon chuyen dong cua clip."""\n    return _motion(read_frames(path, size=160))\n\n\ndef dhash(frame, hash_size=8):\n    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)\n    small = cv2.resize(gray, (hash_size + 1, hash_size))\n    bits = (small[:, 1:] > small[:, :-1]).flatten()\n    return int("".join("1" if bit else "0" for bit in bits), 2)\n\n\ndef find_duplicates(hashes):\n    groups = defaultdict(list)\n    for video, value in hashes.items():\n        groups[value].append(video)\n    return [sorted(videos) for videos in groups.values() if len(videos) > 1]\n\n\ndef summarize_videos(df):\n    return df.groupby(["split", "label"]).agg(clips=("duration_s", "size"), duration_mean=("duration_s", "mean"))\n\n\ndef analyze_clips(df, root, n_samples=3, detector=None):\n    """Tinh chat luong anh, chuyen dong, dHash va so nguoi cho tung clip."""\n    rows = []\n    for row in df.itertuples():\n        frames = read_frames(Path(root) / row.relpath)\n        chosen = [frames[i] for i in sample_indices(len(frames), n_samples)]\n        quality = pd.DataFrame([frame_quality(f) for f in chosen]).mean().to_dict()\n        record = {\n            "video": row.video,\n            "label": row.label,\n            "split": row.split,\n            "motion": _motion([cv2.resize(f, (160, 160)) for f in frames]),\n            "dhash": dhash(frames[len(frames) // 2]),\n            **quality,\n        }\n        if detector:\n            detections = [detector(f) for f in chosen]\n            areas = [\n                (b[2] - b[0]) * (b[3] - b[1]) / (f.shape[0] * f.shape[1])\n                for boxes, f in zip(detections, chosen)\n                for b in boxes\n            ]\n            record["persons_mean"] = float(np.mean([len(boxes) for boxes in detections]))\n            record["no_person_rate"] = float(np.mean([len(boxes) == 0 for boxes in detections]))\n            record["person_area_mean"] = float(np.mean(areas)) if areas else 0.0\n        rows.append(record)\n    return pd.DataFrame(rows)\n\n\ndef plot_by_label(clips, column, out_path):\n    fig, ax = plt.subplots(figsize=(6, 3.5))\n    bins = np.histogram_bin_edges(clips[column], bins=30)\n    for label, color in LABEL_COLORS.items():\n        ax.hist(clips.loc[clips["label"] == label, column], bins=bins, alpha=0.55, label=label, color=color)\n    ax.set(xlabel=column, ylabel="So clip", title=f"Phan bo {column} theo lop")\n    ax.legend()\n    fig.tight_layout()\n    fig.savefig(out_path, dpi=150)\n    plt.close(fig)\n\n\ndef plot_samples(df, root, out_path, per_class=4, seed=42):\n    fig, axes = plt.subplots(2, per_class, figsize=(per_class * 2.5, 5.2), squeeze=False)\n    for axes_row, label in zip(axes, LABEL_COLORS):\n        subset = df[df["label"] == label]\n        picks = subset.sample(min(per_class, len(subset)), random_state=seed)\n        for ax in axes_row:\n            ax.axis("off")\n        for ax, row in zip(axes_row, picks.itertuples()):\n            frames = read_frames(Path(root) / row.relpath)\n            ax.imshow(cv2.cvtColor(frames[len(frames) // 2], cv2.COLOR_BGR2RGB))\n            ax.set_title(label, fontsize=9)\n    fig.tight_layout()\n    fig.savefig(out_path, dpi=150)\n    plt.close(fig)\n\n\ndef run_eda(root, splits_csv, out_dir, detector=None):\n    out = Path(out_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    df = pd.read_csv(splits_csv)\n\n    summary = summarize_videos(df)\n    summary.to_csv(out / "summary.csv")\n    df[["fps", "width", "height", "n_frames"]].describe().to_csv(out / "video_properties.csv")\n\n    clips = analyze_clips(df, root, detector=detector)\n    clips.to_csv(out / "clips.csv", index=False)\n\n    duplicates = find_duplicates(dict(zip(clips["video"] + "/" + clips["label"], clips["dhash"])))\n    pd.DataFrame({"videos": [" | ".join(group) for group in duplicates]}).to_csv(out / "duplicates.csv", index=False)\n\n    columns = ["motion", "brightness", "contrast", "sharpness"]\n    if detector:\n        columns += ["persons_mean", "no_person_rate", "person_area_mean"]\n    for column in columns:\n        plot_by_label(clips, column, out / f"{column}.png")\n    plot_samples(df, root, out / "samples.png")\n\n    by_label = clips.groupby("label")[columns].mean()\n    by_label.to_csv(out / "by_label.csv")\n    return {"summary": summary, "clips": clips, "by_label": by_label, "duplicates": duplicates}\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--root", required=True)\n    parser.add_argument("--splits", default="splits/splits.csv")\n    parser.add_argument("--out", default="results/eda")\n    parser.add_argument("--yolo-weights", help="Vd yolo11n.pt; bo trong de khong thong ke so nguoi")\n    args = parser.parse_args()\n\n    detector = None\n    if args.yolo_weights:\n        from crop_persons import make_yolo_detector\n\n        detector = make_yolo_detector(args.yolo_weights)\n    result = run_eda(args.root, args.splits, args.out, detector)\n    print(result["summary"])\n    print(result["by_label"])\n    print(f"So nhom clip trung nhau: {len(result[\'duplicates\'])}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8');

In [ ]:
(SRC / 'evaluate.py').write_text('"""Danh gia mot mo hinh tu checkpoint o cap clip: chon nguong tren val, tinh chi so tren test.\n\n    python src/evaluate.py --model m1\n"""\nimport argparse\nimport json\nimport time\nfrom pathlib import Path\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import roc_curve\nfrom torch.utils.data import DataLoader\n\nfrom fight_data import ClipDataset, FrameDataset, list_clips\nfrom metrics import aggregate_clips, choose_threshold, clip_metrics, comparison_table, errors_table\nfrom models import CNNLSTM, build_cnn\nfrom prepare_data import read_image\nfrom train_utils import default_device, predict_clips\n\n\ndef per_second(fn, repeats=30):\n    fn()  # lan dau thuong cham hon do khoi tao\n    start = time.perf_counter()\n    for _ in range(repeats):\n        fn()\n    return repeats / (time.perf_counter() - start)\n\n\ndef load_torch_model(name, ckpt, device):\n    checkpoint = torch.load(ckpt, map_location=device, weights_only=True)\n    config = checkpoint["config"]\n    if name == "m1":\n        model = build_cnn(config["arch"], pretrained=False)\n    else:\n        model = CNNLSTM(config["arch"], config["hidden_size"], pretrained=False)\n    model.load_state_dict(checkpoint["state_dict"])\n    return model.to(device).eval()\n\n\ndef evaluate_torch(args, device):\n    model = load_torch_model(args.model, args.ckpt, device)\n    dataset_cls = FrameDataset if args.model == "m1" else ClipDataset\n\n    def predict(split):\n        loader = DataLoader(dataset_cls(args.frames, split, args.image_size), args.batch_size)\n        return predict_clips(model, loader, device)\n\n    sample = dataset_cls(args.frames, "test", args.image_size)[0][0].unsqueeze(0).to(device)\n\n    @torch.no_grad()\n    def forward():\n        model(sample)\n\n    frames_per_call = sample.shape[1] if args.model == "m3" else 1\n    return predict("val"), predict("test"), per_second(forward) * frames_per_call\n\n\ndef evaluate_yolo(args, device):\n    from ultralytics import YOLO\n\n    model = YOLO(str(args.ckpt))\n    fight = next(i for i, name in model.names.items() if name == "Fight")\n\n    def predict(split):\n        rows = []\n        for clip in list_clips(args.frames, split):\n            results = model.predict(\n                [str(p) for p in clip["frames"]], imgsz=args.image_size, device=device, verbose=False\n            )\n            rows += [\n                {"video": clip["video"], "label": clip["label"], "prob": float(r.probs.data[fight])} for r in results\n            ]\n        return aggregate_clips(pd.DataFrame(rows))\n\n    image = read_image(list_clips(args.frames, "test")[0]["frames"][0])\n    fps = per_second(lambda: model.predict(image, imgsz=args.image_size, device=device, verbose=False))\n    return predict("val"), predict("test"), fps\n\n\ndef add_detector_time(fps, args, device):\n    """M3 con phai chay YOLO phat hien nguoi tren moi frame, nen tinh ca thoi gian do."""\n    if not args.yolo_weights:\n        print("CANH BAO: khong co --yolo-weights, FPS cua M3 chua tinh buoc phat hien nguoi")\n        return fps\n    from ultralytics import YOLO\n\n    detector = YOLO(str(args.yolo_weights))\n    image = read_image(list_clips(args.full_frames, "test")[0]["frames"][0])\n    detector_fps = per_second(lambda: detector.predict(image, classes=[0], device=device, verbose=False))\n    return 1 / (1 / fps + 1 / detector_fps)\n\n\ndef save_plots(clips, metrics, out):\n    fig, ax = plt.subplots(figsize=(3.6, 3.2))\n    matrix = np.array([[metrics["tn"], metrics["fp"]], [metrics["fn"], metrics["tp"]]])\n    ax.imshow(matrix, cmap="Blues")\n    for (row, col), value in np.ndenumerate(matrix):\n        ax.text(col, row, value, ha="center", va="center", color="white" if value > matrix.max() / 2 else "black")\n    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["NonFight", "Fight"], yticklabels=["NonFight", "Fight"],\n           xlabel="Du doan", ylabel="Thuc te")\n    fig.tight_layout()\n    fig.savefig(out / "confusion_matrix.png", dpi=150)\n    plt.close(fig)\n\n    fpr, tpr, _ = roc_curve(clips["label"], clips["prob"])\n    fig, ax = plt.subplots(figsize=(3.6, 3.2))\n    ax.plot(fpr, tpr, label=f"AUC = {metrics[\'roc_auc\']:.3f}")\n    ax.plot([0, 1], [0, 1], linestyle="--", color="gray")\n    ax.set(xlabel="False positive rate", ylabel="Recall")\n    ax.legend()\n    fig.tight_layout()\n    fig.savefig(out / "roc.png", dpi=150)\n    plt.close(fig)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--model", required=True, choices=["m1", "m2", "m3"])\n    parser.add_argument("--ckpt")\n    parser.add_argument("--frames", help="Mac dinh data/crops voi m3, data/frames voi m1 va m2")\n    parser.add_argument("--full-frames", default="data/frames", help="Frame goc, de do toc do phat hien nguoi cua m3")\n    parser.add_argument("--results", default="results")\n    parser.add_argument("--image-size", type=int, default=224)\n    parser.add_argument("--batch-size", type=int, default=16)\n    parser.add_argument("--yolo-weights", help="Trong so phat hien nguoi, chi dung voi m3")\n    args = parser.parse_args()\n\n    out = Path(args.results) / args.model\n    out.mkdir(parents=True, exist_ok=True)\n    args.ckpt = args.ckpt or out / "best.pt"\n    args.frames = args.frames or ("data/crops" if args.model == "m3" else "data/frames")\n    device = default_device()\n\n    if args.model == "m2":\n        val_clips, test_clips, fps = evaluate_yolo(args, device)\n    else:\n        val_clips, test_clips, fps = evaluate_torch(args, device)\n    if args.model == "m3":\n        fps = add_detector_time(fps, args, device)\n\n    threshold = choose_threshold(val_clips["label"], val_clips["prob"])\n    metrics = clip_metrics(test_clips["label"], test_clips["prob"], threshold)\n    metrics.update(\n        model=args.model,\n        fps=round(fps, 1),\n        size_mb=round(Path(args.ckpt).stat().st_size / 1e6, 1),\n        device=torch.cuda.get_device_name(0) if device == "cuda" else "cpu",\n        test_clips=len(test_clips),\n    )\n    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))\n    test_clips.to_csv(out / "preds_test.csv", index=False)\n    errors_table(test_clips, threshold).to_csv(out / "errors.csv", index=False)\n    save_plots(test_clips, metrics, out)\n\n    table = comparison_table(args.results)\n    table.to_csv(Path(args.results) / "comparison.csv", index=False)\n    print(json.dumps(metrics, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8');

In [ ]:
(SRC / 'fight_data.py').write_text('"""Dataset cho M1 (tung frame) va M3 (chuoi frame). Nhan: Fight = 1, NonFight = 0."""\nimport random\nfrom pathlib import Path\n\nimport cv2\nimport torch\nfrom torch.utils.data import Dataset\nfrom torchvision.transforms import functional as TF\n\nfrom prepare_data import read_image\n\nIMAGENET_MEAN = [0.485, 0.456, 0.406]\nIMAGENET_STD = [0.229, 0.224, 0.225]\n\n\ndef list_clips(root, split):\n    clips = []\n    for label_name, label in (("Fight", 1), ("NonFight", 0)):\n        for clip_dir in sorted((Path(root) / split / label_name).glob("*")):\n            frames = sorted(clip_dir.glob("*.jpg"))\n            if frames:\n                clips.append({"video": clip_dir.name, "label": label, "frames": frames})\n    return clips\n\n\ndef sample_augmentation(image_size):\n    """Rut tham so mot lan de co the ap dung giong nhau cho moi frame trong clip."""\n    crop = random.randint(int(image_size * 0.85), image_size)\n    return {\n        "flip": random.random() < 0.5,\n        "brightness": random.uniform(0.8, 1.2),\n        "contrast": random.uniform(0.8, 1.2),\n        "saturation": random.uniform(0.8, 1.2),\n        "crop": crop,\n        "top": random.random(),\n        "left": random.random(),\n    }\n\n\ndef load_tensor(path, image_size, params=None):\n    image = cv2.cvtColor(read_image(path), cv2.COLOR_BGR2RGB)\n    image = cv2.resize(image, (image_size, image_size))\n    tensor = torch.from_numpy(image).permute(2, 0, 1).float() / 255\n    if params:\n        crop = params["crop"]\n        top = int(params["top"] * (image_size - crop))\n        left = int(params["left"] * (image_size - crop))\n        tensor = TF.resized_crop(tensor, top, left, crop, crop, [image_size, image_size], antialias=True)\n        if params["flip"]:\n            tensor = TF.hflip(tensor)\n        tensor = TF.adjust_brightness(tensor, params["brightness"])\n        tensor = TF.adjust_contrast(tensor, params["contrast"])\n        tensor = TF.adjust_saturation(tensor, params["saturation"])\n    return TF.normalize(tensor, IMAGENET_MEAN, IMAGENET_STD)\n\n\nclass FrameDataset(Dataset):\n    def __init__(self, root, split, image_size=224, augment=False):\n        self.items = [\n            (frame, clip["label"], clip["video"]) for clip in list_clips(root, split) for frame in clip["frames"]\n        ]\n        self.image_size = image_size\n        self.augment = augment\n\n    def __len__(self):\n        return len(self.items)\n\n    def __getitem__(self, index):\n        path, label, video = self.items[index]\n        params = sample_augmentation(self.image_size) if self.augment else None\n        return load_tensor(path, self.image_size, params), label, video\n\n\nclass ClipDataset(Dataset):\n    def __init__(self, root, split, image_size=224, augment=False):\n        self.clips = list_clips(root, split)\n        self.image_size = image_size\n        self.augment = augment\n\n    def __len__(self):\n        return len(self.clips)\n\n    def __getitem__(self, index):\n        clip = self.clips[index]\n        params = sample_augmentation(self.image_size) if self.augment else None\n        frames = [load_tensor(path, self.image_size, params) for path in clip["frames"]]\n        return torch.stack(frames), clip["label"], clip["video"]\n', encoding='utf-8');

In [ ]:
(SRC / 'metrics.py').write_text('"""Chi so cap clip, dung chung cho ca ba mo hinh. Lop duong (1) la Fight."""\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import confusion_matrix, fbeta_score, roc_auc_score\n\n\ndef aggregate_clips(frames):\n    """Gop xac suat tung frame thanh mot xac suat cho moi clip."""\n    return frames.groupby(["video", "label"], as_index=False)["prob"].mean()\n\n\ndef clip_metrics(y, p, threshold=0.5):\n    y, p = np.asarray(y), np.asarray(p)\n    pred = (p >= threshold).astype(int)\n    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()\n    precision = tp / (tp + fp) if tp + fp else 0.0\n    recall = tp / (tp + fn) if tp + fn else 0.0\n    return {\n        "threshold": float(threshold),\n        "accuracy": float((tp + tn) / len(y)),\n        "precision": float(precision),\n        "recall": float(recall),\n        "f1": float(2 * precision * recall / (precision + recall)) if precision + recall else 0.0,\n        "roc_auc": float(roc_auc_score(y, p)) if len(set(y)) == 2 else float("nan"),\n        "tp": int(tp),\n        "fp": int(fp),\n        "fn": int(fn),\n        "tn": int(tn),\n    }\n\n\ndef choose_threshold(y, p):\n    """Chon nguong tot nhat theo F2 de uu tien Recall. Chi goi tren tap validation."""\n    candidates = np.unique(p)\n    scores = [fbeta_score(y, p >= t, beta=2, zero_division=0) for t in candidates]\n    # Khi hoa diem, lay nguong cao nhat de bot bao dong gia.\n    return float(candidates[len(scores) - 1 - int(np.argmax(scores[::-1]))])\n\n\ndef errors_table(clips, threshold):\n    pred = (clips["prob"] >= threshold).astype(int)\n    wrong = clips[pred != clips["label"]].copy()\n    wrong["error"] = np.where(wrong["label"] == 1, "FN", "FP")\n    return wrong.sort_values("prob", ascending=False)\n\n\ndef comparison_table(results_dir):\n    rows = [json.loads(p.read_text()) for p in Path(results_dir).glob("*/metrics.json")]\n    return pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)\n', encoding='utf-8');

In [ ]:
(SRC / 'models.py').write_text('"""M1: CNN phan loai frame. M3: CNN + LSTM phan loai chuoi frame."""\nimport torch\nfrom torch import nn\nfrom torchvision import models as tv\n\n\ndef build_backbone(arch, pretrained=True):\n    """Tra ve mang trich dac trung va so chieu dac trung."""\n    if arch == "resnet18":\n        net = tv.resnet18(weights=tv.ResNet18_Weights.DEFAULT if pretrained else None)\n        dim = net.fc.in_features\n        net.fc = nn.Identity()\n    elif arch == "mobilenet_v2":\n        net = tv.mobilenet_v2(weights=tv.MobileNet_V2_Weights.DEFAULT if pretrained else None)\n        dim = net.last_channel\n        net.classifier = nn.Identity()\n    else:\n        raise ValueError(f"Kien truc khong ho tro: {arch}")\n    return net, dim\n\n\nclass FrameCNN(nn.Module):\n    def __init__(self, arch="resnet18", pretrained=True, dropout=0.3):\n        super().__init__()\n        self.backbone, dim = build_backbone(arch, pretrained)\n        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(dim, 1))\n\n    def forward(self, x):\n        return self.head(self.backbone(x)).squeeze(-1)\n\n\ndef build_cnn(arch="resnet18", pretrained=True):\n    return FrameCNN(arch, pretrained)\n\n\nclass CNNLSTM(nn.Module):\n    def __init__(self, arch="resnet18", hidden_size=256, pretrained=True, freeze_backbone=False, dropout=0.3):\n        super().__init__()\n        self.backbone, dim = build_backbone(arch, pretrained)\n        self.freeze_backbone = freeze_backbone\n        if freeze_backbone:\n            self.backbone.requires_grad_(False)\n        self.lstm = nn.LSTM(dim, hidden_size, batch_first=True)\n        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_size, 1))\n\n    def train(self, mode=True):\n        super().train(mode)\n        if self.freeze_backbone:\n            self.backbone.eval()  # giu nguyen thong ke BatchNorm khi dong bang\n        return self\n\n    def forward(self, clips):\n        batch, steps = clips.shape[:2]\n        with torch.set_grad_enabled(self.training and not self.freeze_backbone):\n            features = self.backbone(clips.flatten(0, 1))\n        _, (hidden, _) = self.lstm(features.view(batch, steps, -1))\n        return self.head(hidden[-1]).squeeze(-1)\n', encoding='utf-8');

In [ ]:
(SRC / 'prepare_data.py').write_text('"""Kiem tra RWF-2000, chia train/val/test theo video va trich xuat frame.\n\n    python src/prepare_data.py --root D:/RWF-2000 --splits splits/splits.csv --frames data/frames\n"""\nimport argparse\nimport re\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedGroupKFold\n\nLABELS = ("Fight", "NonFight")\nVIDEO_EXTS = {".avi", ".mp4", ".mov", ".mkv"}\n\n\ndef read_image(path):\n    """Doc anh, ho tro ca duong dan co dau tieng Viet tren Windows."""\n    return cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_COLOR)\n\n\ndef write_image(path, image):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    ok, buffer = cv2.imencode(".jpg", image)\n    if not ok:\n        raise ValueError(f"Khong ma hoa duoc anh: {path}")\n    buffer.tofile(str(path))\n\n\ndef scan_dataset(root):\n    """Liet ke clip theo cau truc <root>/{train,val}/{Fight,NonFight}/*."""\n    root = Path(root)\n    rows = []\n    for source_split in ("train", "val"):\n        for label in LABELS:\n            folder = root / source_split / label\n            if not folder.is_dir():\n                raise FileNotFoundError(f"Thieu thu muc: {folder}")\n            for path in sorted(folder.iterdir()):\n                if path.suffix.lower() in VIDEO_EXTS:\n                    rows.append(\n                        {\n                            "video": path.stem,\n                            "label": label,\n                            "source_split": source_split,\n                            "relpath": path.relative_to(root).as_posix(),\n                        }\n                    )\n    return pd.DataFrame(rows)\n\n\ndef source_id(video):\n    """Bo chi so clip o cuoi ten: \'abc_12\' -> \'abc\'. Cac clip cung nguon co the rat giong nhau."""\n    return re.sub(r"_\\d+$", "", video)\n\n\ndef make_splits(df, val_fraction=0.1, seed=42):\n    """Giu \'val\' goc lam test; tach val tu \'train\' goc, phan tang va khong tach clip cung nguon."""\n    out = df.copy()\n    out["split"] = "train"\n    out.loc[out["source_split"] == "val", "split"] = "test"\n\n    pool = out[out["split"] == "train"]\n    n_val = round(val_fraction * len(out))\n    folds = StratifiedGroupKFold(n_splits=max(2, round(len(pool) / n_val)), shuffle=True, random_state=seed)\n    _, val_positions = next(folds.split(pool, pool["label"], groups=pool["video"].map(source_id)))\n    out.loc[pool.index[val_positions], "split"] = "val"\n    return out\n\n\ndef check_no_leak(df):\n    """Bao loi neu ro ri trong train/val; tra ve cac nguon dung chung voi test de ghi vao bao cao."""\n    per_video = df.groupby(["video", "label"])["split"].nunique()\n    repeated = per_video[per_video > 1]\n    if len(repeated):\n        raise ValueError(f"Video nam o nhieu tap: {sorted(v for v, _ in repeated.index)[:10]}")\n\n    sources = df["video"].map(source_id)\n    develop = df["split"].isin(["train", "val"])\n    per_source = df[develop].groupby(sources[develop])["split"].nunique()\n    shared = per_source[per_source > 1]\n    if len(shared):\n        raise ValueError(f"Nguon video nam o ca train va val: {sorted(shared.index)[:10]}")\n\n    test_sources = set(sources[df["split"] == "test"])\n    return sorted(test_sources & set(sources[develop]))\n\n\ndef sample_indices(n_frames, k):\n    """Chon k vi tri cach deu nhau, lay diem giua moi doan."""\n    return ((np.arange(k) + 0.5) * n_frames / k).astype(int)\n\n\ndef probe_video(path):\n    capture = cv2.VideoCapture(str(path))\n    info = {"readable": False, "fps": 0.0, "n_frames": 0, "width": 0, "height": 0, "duration_s": 0.0}\n    if capture.isOpened():\n        readable, _ = capture.read()\n        fps = capture.get(cv2.CAP_PROP_FPS)\n        n_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))\n        info = {\n            "readable": bool(readable) and n_frames > 0,\n            "fps": fps,\n            "n_frames": n_frames,\n            "width": int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),\n            "height": int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)),\n            "duration_s": n_frames / fps if fps > 0 else 0.0,\n        }\n    capture.release()\n    return info\n\n\ndef read_frames(path, size=None):\n    """Doc toan bo frame vi so frame trong metadata co the sai; size la canh anh vuong, None giu nguyen."""\n    capture = cv2.VideoCapture(str(path))\n    frames = []\n    while True:\n        ok, frame = capture.read()\n        if not ok:\n            break\n        frames.append(cv2.resize(frame, (size, size)) if size else frame)\n    capture.release()\n    return frames\n\n\ndef extract_frames(video_path, out_dir, num_frames=16, size=None):\n    frames = read_frames(video_path, size)\n    if not frames:\n        return 0\n    for position, index in enumerate(sample_indices(len(frames), num_frames)):\n        write_image(Path(out_dir) / f"frame_{position:02d}.jpg", frames[index])\n    return num_frames\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--root", required=True, help="Thu muc RWF-2000")\n    parser.add_argument("--splits", default="splits/splits.csv")\n    parser.add_argument("--frames", default="data/frames")\n    parser.add_argument("--num-frames", type=int, default=16)\n    parser.add_argument("--val-fraction", type=float, default=0.1)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n\n    root = Path(args.root)\n    index = scan_dataset(root)\n    probes = pd.DataFrame([probe_video(root / rel) for rel in index["relpath"]])\n    index = pd.concat([index, probes], axis=1)\n\n    bad = index[~index["readable"]]\n    if len(bad):\n        print(f"Bo {len(bad)} clip khong doc duoc: {bad[\'relpath\'].tolist()}")\n    index = index[index["readable"]].reset_index(drop=True)\n\n    splits = make_splits(index, args.val_fraction, args.seed)\n    overlap = check_no_leak(splits)\n    if overlap:\n        print(f"CANH BAO: {len(overlap)} nguon video xuat hien o ca test va train/val: {overlap[:10]}")\n\n    splits_path = Path(args.splits)\n    splits_path.parent.mkdir(parents=True, exist_ok=True)\n    splits.to_csv(splits_path, index=False)\n    print(splits.groupby(["split", "label"]).size())\n\n    for row in splits.itertuples():\n        out_dir = Path(args.frames) / row.split / row.label / row.video\n        if len(list(out_dir.glob("*.jpg"))) < args.num_frames:\n            extract_frames(root / row.relpath, out_dir, args.num_frames)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8');

In [ ]:
(SRC / 'train_cnn.py').write_text('"""M1: CNN baseline tren tung frame.  python src/train_cnn.py"""\nfrom fight_data import FrameDataset\nfrom models import build_cnn\nfrom train_utils import run_training, train_parser\n\n\ndef main():\n    args = train_parser("data/frames", "results/m1", batch_size=32).parse_args()\n    model = build_cnn(args.arch, pretrained=not args.no_pretrained)\n    run_training(args, model, FrameDataset, {"arch": args.arch})\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8');

In [ ]:
(SRC / 'train_cnn_lstm.py').write_text('"""M3: CNN + LSTM tren chuoi frame da crop nguoi (chay crop_persons.py truoc).  python src/train_cnn_lstm.py"""\nfrom fight_data import ClipDataset\nfrom models import CNNLSTM\nfrom train_utils import run_training, train_parser\n\n\ndef main():\n    parser = train_parser("data/crops", "results/m3", batch_size=8)\n    parser.add_argument("--hidden-size", type=int, default=256)\n    parser.add_argument("--freeze-backbone", action="store_true")\n    args = parser.parse_args()\n    model = CNNLSTM(args.arch, args.hidden_size, not args.no_pretrained, args.freeze_backbone)\n    run_training(args, model, ClipDataset, {"arch": args.arch, "hidden_size": args.hidden_size})\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8');

In [ ]:
(SRC / 'train_utils.py').write_text('"""Vong huan luyen dung chung cho M1 va M3: dung som theo F1 cap clip tren validation."""\nimport argparse\nimport json\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom metrics import aggregate_clips, clip_metrics\n\n\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n\n\ndef default_device():\n    return "cuda" if torch.cuda.is_available() else "cpu"\n\n\n@torch.no_grad()\ndef predict_clips(model, loader, device):\n    model.eval()\n    rows = []\n    for inputs, labels, videos in loader:\n        probs = torch.sigmoid(model(inputs.to(device))).cpu().tolist()\n        rows += [\n            {"video": video, "label": int(label), "prob": prob} for video, label, prob in zip(videos, labels, probs)\n        ]\n    return aggregate_clips(pd.DataFrame(rows))\n\n\ndef fit(model, train_loader, val_loader, out_dir, epochs, patience, lr, device, weight_decay=1e-4, config=None):\n    out = Path(out_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    model.to(device)\n    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay)\n    criterion = torch.nn.BCEWithLogitsLoss()\n    history, best_f1, stale = [], -1.0, 0\n\n    for epoch in range(1, epochs + 1):\n        model.train()\n        total_loss, seen = 0.0, 0\n        for inputs, labels, _ in train_loader:\n            inputs, labels = inputs.to(device), labels.float().to(device)\n            optimizer.zero_grad()\n            loss = criterion(model(inputs), labels)\n            loss.backward()\n            optimizer.step()\n            total_loss += loss.item() * len(labels)\n            seen += len(labels)\n\n        clips = predict_clips(model, val_loader, device)\n        val = clip_metrics(clips["label"], clips["prob"])\n        history.append({"epoch": epoch, "train_loss": total_loss / seen, "val_f1": val["f1"], "val_acc": val["accuracy"]})\n        pd.DataFrame(history).to_csv(out / "history.csv", index=False)\n        print(history[-1], flush=True)\n\n        if val["f1"] > best_f1:\n            best_f1, stale = val["f1"], 0\n            torch.save({"state_dict": model.state_dict(), "config": config or {}}, out / "best.pt")\n        else:\n            stale += 1\n            if stale >= patience:\n                break\n    return history\n\n\ndef train_parser(default_frames, default_out, batch_size):\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--frames", default=default_frames)\n    parser.add_argument("--out", default=default_out)\n    parser.add_argument("--arch", default="resnet18", choices=["resnet18", "mobilenet_v2"])\n    parser.add_argument("--no-pretrained", action="store_true")\n    parser.add_argument("--image-size", type=int, default=224)\n    parser.add_argument("--epochs", type=int, default=30)\n    parser.add_argument("--patience", type=int, default=5)\n    parser.add_argument("--batch-size", type=int, default=batch_size)\n    parser.add_argument("--lr", type=float, default=1e-4)\n    parser.add_argument("--num-workers", type=int, default=2)\n    parser.add_argument("--seed", type=int, default=42)\n    return parser\n\n\ndef run_training(args, model, dataset_cls, config):\n    set_seed(args.seed)\n    Path(args.out).mkdir(parents=True, exist_ok=True)\n    (Path(args.out) / "config.json").write_text(json.dumps(vars(args), indent=2))\n    train_set = dataset_cls(args.frames, "train", args.image_size, augment=True)\n    val_set = dataset_cls(args.frames, "val", args.image_size)\n    train_loader = DataLoader(train_set, args.batch_size, shuffle=True, num_workers=args.num_workers)\n    val_loader = DataLoader(val_set, args.batch_size, num_workers=args.num_workers)\n    return fit(model, train_loader, val_loader, args.out, args.epochs, args.patience, args.lr, default_device(), config=config)\n', encoding='utf-8');

In [ ]:
(SRC / 'train_yolo_cls.py').write_text('"""M2: YOLO11 phan loai tren tung frame.  python src/train_yolo_cls.py\n\nUltralytics dung som theo top-1 cap frame, khac M1/M3 (F1 cap clip). Hai lop can bang nen chap nhan duoc;\ndanh gia cuoi van o cap clip qua evaluate.py.\n"""\nimport argparse\nimport shutil\nfrom pathlib import Path\n\nfrom ultralytics import YOLO, settings\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--frames", default="data/frames")\n    parser.add_argument("--out", default="results/m2")\n    parser.add_argument("--weights", default="yolo11n-cls.pt")\n    parser.add_argument("--image-size", type=int, default=224)\n    parser.add_argument("--epochs", type=int, default=30)\n    parser.add_argument("--patience", type=int, default=5)\n    parser.add_argument("--batch-size", type=int, default=32)\n    parser.add_argument("--workers", type=int, default=2)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n\n    settings.update({"sync": False})\n    out = Path(args.out).resolve()\n    model = YOLO(str(args.weights))\n    model.train(\n        data=str(Path(args.frames).resolve()),\n        imgsz=args.image_size,\n        epochs=args.epochs,\n        patience=args.patience,\n        batch=args.batch_size,\n        workers=args.workers,\n        seed=args.seed,\n        project=str(out),\n        name="train",\n        exist_ok=True,\n        plots=False,\n    )\n    shutil.copy(Path(model.trainer.best), out / "best.pt")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8');

## 1. Kiểm tra và chia dữ liệu, trích frame

In [ ]:
run('prepare_data.py', '--root', ROOT, '--frames', DATA / 'frames')


## 2. EDA

In [ ]:
run('eda.py', '--root', ROOT, '--yolo-weights', 'yolo11n.pt')
from IPython.display import Image, display
import pandas as pd
display(pd.read_csv('results/eda/by_label.csv'))
for name in ['samples', 'motion', 'persons_mean']:
    display(Image(filename=f'results/eda/{name}.png'))


## 3. M1: CNN baseline

In [ ]:
run('train_cnn.py', '--frames', DATA / 'frames', '--epochs', EPOCHS, '--image-size', IMAGE_SIZE, *EXTRA)


## 4. M2: YOLO11-cls

In [ ]:
run('train_yolo_cls.py', '--frames', DATA / 'frames', '--epochs', EPOCHS, '--image-size', IMAGE_SIZE)


## 5. M3: crop người bằng YOLO11, sau đó CNN + LSTM

In [ ]:
run('crop_persons.py', '--frames', DATA / 'frames', '--out', DATA / 'crops', '--size', IMAGE_SIZE)
run('train_cnn_lstm.py', '--frames', DATA / 'crops', '--epochs', EPOCHS, '--image-size', IMAGE_SIZE, *EXTRA)


## 6. Đánh giá trên tập test

Ngưỡng được chọn trên validation. Bảng cuối là so sánh ba mô hình.

In [ ]:
common = ['--image-size', IMAGE_SIZE]
run('evaluate.py', '--model', 'm1', '--frames', DATA / 'frames', *common)
run('evaluate.py', '--model', 'm2', '--frames', DATA / 'frames', *common)
run('evaluate.py', '--model', 'm3', '--frames', DATA / 'crops', '--full-frames', DATA / 'frames',
    '--yolo-weights', 'yolo11n.pt', *common)
pd.read_csv('results/comparison.csv')


## 7. Đóng gói kết quả

Tải `results.zip` ở tab Output, gồm cả checkpoint và `splits.csv`.

In [ ]:
shutil.copy('splits/splits.csv', 'results/splits.csv')
shutil.make_archive('results', 'zip', 'results')
print('Da tao results.zip')
